In [6]:
from statsbombpy import sb
import warnings
from statsbombpy.api_client import NoAuthWarning

warnings.filterwarnings("ignore", category=NoAuthWarning)


def _is_true(val):
    """Safe boolean — treats NaN, None, and False all as False.
    NaN == True is False in Python, so this is safe."""
    return val == True

matches = sb.matches(competition_id=11, season_id=26)
events  = sb.events(match_id=matches.iloc[0]['match_id'])
passes  = events[events['type'] == 'Pass']

prog_count = 0
back_count = 0
none_count = 0
error_count = 0

for _, row in passes.iterrows():
    d = row.to_dict()
    try:
        start = d.get('location')
        end   = d.get('pass_end_location')

        # Print the first 3 passes to see exact types
        if prog_count + back_count + none_count < 3:
            print(f"location type: {type(start)}, value: {start}")
            print(f"end_location type: {type(end)}, value: {end}")
            print(f"pass_switch: {repr(d.get('pass_switch'))}")
            print("---")

        if start is None or end is None:
            none_count += 1
            continue

        diff = end[0] - start[0]
        if diff >= 10:
            prog_count += 1
        else:
            none_count += 1

    except Exception as e:
        error_count += 1
        print(f"Error: {e}  |  start={start}  end={end}")

print(f"\nProgressive passes found: {prog_count}")
print(f"Skipped (None/backward): {none_count}")
print(f"Errors: {error_count}")

location type: <class 'list'>, value: [61.0, 40.1]
end_location type: <class 'list'>, value: [57.6, 36.8]
pass_switch: nan
---
location type: <class 'list'>, value: [55.3, 37.5]
end_location type: <class 'list'>, value: [43.9, 41.2]
pass_switch: nan
---
location type: <class 'list'>, value: [43.9, 41.2]
end_location type: <class 'list'>, value: [43.9, 15.2]
pass_switch: nan
---

Progressive passes found: 324
Skipped (None/backward): 697
Errors: 0


In [2]:
from gensim.models import Word2Vec
import pickle

model = Word2Vec.load('../models/action2vec.model')

# Word2Vec vocab counts (how often each token appeared in training)
print("Token frequencies across all possession chains:\n")
for token, idx in sorted(model.wv.key_to_index.items()):
    count = model.wv.get_vecattr(token, 'count')
    flag = " <-- RARE, consider merging" if count < 500 else ""
    print(f"  {token:<20} {count:>8,}{flag}")

Token frequencies across all possession chains:

  CARRY                 191,249
  CARRY_PROG             53,048
  CARRY_WIDE            169,731
  CARRY_WIDE_PROG        46,604
  CLEARANCE              21,321
  DRIBBLE_L               7,872
  DRIBBLE_W              10,901
  FOUL_W                 13,652
  INT                     3,438
  PASS_BACK              13,968
  PASS_CROSS             13,448
  PASS_PROG             150,841
  PRESS                 165,162
  SHOT_OFF                2,920
  SHOT_OFF_BOX            3,458
  SHOT_ON                 1,203
  SHOT_ON_BOX             3,045


In [8]:
import pickle
from collections import Counter

with open('../data/processed/corpus.pkl', 'rb') as f:
    corpus = pickle.load(f)

all_tokens = [token for chain in corpus for token in chain]
print(Counter(all_tokens))

Counter({'CARRY': 340079, 'PRESS': 124428, 'PASS_PROG': 115292, 'CLEARANCE': 17452, 'PASS_BACK': 10807, 'PASS_CROSS': 10746, 'FOUL_W': 10726, 'DRIBBLE_W': 8352, 'DRIBBLE_L': 6205, 'SHOT_OFF': 4894, 'SHOT_ON': 3127, 'INT': 2681})


In [3]:
import numpy as np
from gensim.models import Word2Vec

model = Word2Vec.load('../models/action2vec.model')
print("Word2Vec vector size:", model.vector_size)

matrix = np.load('../models/player_matrix.npy')
print("Player matrix shape:", matrix.shape)

emb_32 = np.load('../models/emb_32d.npy')
print("UMAP embedding shape:", emb_32.shape)

Word2Vec vector size: 128
Player matrix shape: (1059, 128)
UMAP embedding shape: (1059, 32)
